In [ ]:
import pickle
import os

print("=" * 50)
print("SAVING MODEL AND SCALER")
print("=" * 50)

os.makedirs('../artifacts', exist_ok=True)

# Save the trained model to a file
with open('../artifacts/model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the scaler to a file too
with open('../artifacts/preprocessor.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Model saved at: artifacts/model.pkl")
print("✅ Scaler saved at: artifacts/preprocessor.pkl")

In [ ]:
print("=" * 50)
print("VISUALIZING ANOMALIES")
print("=" * 50)

plt.figure(figsize=(14, 6))

# Plot the normal closing price line
plt.plot(test_df.index, test_df['Close'], linewidth=2, color='blue', label='Close Price')

# Highlight only the anomaly days with red dots
anomaly_days = test_df[test_df['Is_Anomaly']]
plt.scatter(anomaly_days.index, anomaly_days['Close'], 
            color='red', s=100, label='Anomaly', zorder=5)

plt.title('Stock Price with Detected Anomalies (Test Data)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 50)
print("DETECTING ANOMALIES ON TEST DATA")
print("=" * 50)

# Ask the model to make predictions on the test data
predictions = model.predict(X_test_scaled)

# Add the predictions as a new column in our test dataframe
test_df['Anomaly'] = predictions

# Make a simple True/False column that's easier to read
test_df['Is_Anomaly'] = test_df['Anomaly'] == -1

print("✅ Predictions complete!")
print(f"\nTotal test days: {len(test_df)}")
print(f"Anomalies found: {test_df['Is_Anomaly'].sum()}")
print(f"Normal days: {(~test_df['Is_Anomaly']).sum()}")

print("\n" + "=" * 50)
print("ANOMALOUS DAYS DETECTED")
print("=" * 50)
print(test_df[test_df['Is_Anomaly']][['Close', 'High', 'Low', 'Volume']])

In [ ]:
print("=" * 50)
print("MODEL TRAINING - ISOLATION FOREST")
print("=" * 50)

# Create the Isolation Forest model
model = IsolationForest(
    contamination=0.05,   # We expect ~5% of data to be anomalies
    random_state=42,      # For reproducible results
    n_estimators=100      # Number of trees in the forest
)

# Train the model on scaled training data
model.fit(X_train_scaled)

print("✅ Model trained successfully!")
print(f"\nModel type: {type(model).__name__}")
print(f"Number of trees: {model.n_estimators}")
print(f"Contamination rate: {model.contamination}")

In [ ]:
# Select features for the model
features = ['Close', 'High', 'Low', 'Volume']

print("=" * 50)
print("FEATURE ENGINEERING")
print("=" * 50)
print(f"Features selected: {features}")

# Extract features from train and test data
X_train = train_df[features].values
X_test = test_df[features].values

print(f"\nTrain features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

# Scale the data using StandardScaler
print("\n" + "=" * 50)
print("DATA SCALING")
print("=" * 50)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data scaled successfully!")
print(f"\nScaled train data (first 5 rows):")
print(X_train_scaled[:5])

print(f"\nScaler statistics:")
print(f"Mean: {scaler.mean_}")
print(f"Std Dev: {scaler.scale_}")

In [ ]:
# Load training data from notebooks/data
train_data_path = 'data/train.csv'
test_data_path = 'data/test.csv'

train_df = pd.read_csv(train_data_path, index_col=0, header=[0, 1])
test_df = pd.read_csv(test_data_path, index_col=0, header=[0, 1])

train_df.columns = train_df.columns.get_level_values(0)
test_df.columns = test_df.columns.get_level_values(0)

print("=" * 50)
print("TRAINING DATA LOADED")
print("=" * 50)
print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print("\nFirst 5 rows of training data:")
print(train_df.head())

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import pickle
import os
import sys